In [2]:
import sqlite3
import pandas as pd

# Tạo kết nối CSDL trong bộ nhớ
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng và chèn dữ liệu cho bảng 'student'
cursor.execute('''
CREATE TABLE student (
    student_id INT,
    name TEXT,
    class TEXT,
    course_id TEXT,
    score REAL
)
''')
student_data = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', '12', 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', '34', 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', '20', 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', '24', 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', '24', 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', '34', 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', '20', 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany('INSERT INTO student VALUES (?,?,?,?,?)', student_data)

# Tạo bảng và chèn dữ liệu cho bảng 'course'
cursor.execute('''
CREATE TABLE course (
    id TEXT,
    course_name TEXT
)
''')
course_data = [
    ('12', 'Giai tich'),
    ('34', 'Thong ke'),
    ('26', 'Tin hoc')
]
cursor.executemany('INSERT INTO course VALUES (?,?)', course_data)
conn.commit()

print("Đã khởi tạo dữ liệu thành công!")

Đã khởi tạo dữ liệu thành công!


#Yêu cầu 1

In [3]:
print("--- 1. TÍCH DECARTES (CROSS JOIN) ---")
display(pd.read_sql_query("SELECT * FROM student CROSS JOIN course", conn))

print("\n--- 2. INNER JOIN ---")
display(pd.read_sql_query("SELECT * FROM student INNER JOIN course ON student.course_id = course.id", conn))

print("\n--- 3. LEFT JOIN ---")
display(pd.read_sql_query("SELECT * FROM student LEFT JOIN course ON student.course_id = course.id", conn))

# Lưu ý: RIGHT JOIN và FULL OUTER JOIN được hỗ trợ trong SQLite từ phiên bản 3.39.0 trở lên.
print("\n--- 4. RIGHT JOIN ---")
try:
    display(pd.read_sql_query("SELECT * FROM student RIGHT JOIN course ON student.course_id = course.id", conn))
except Exception as e:
    print("Phiên bản SQLite hiện tại không hỗ trợ RIGHT JOIN:", e)

print("\n--- 5. FULL OUTER JOIN ---")
try:
    display(pd.read_sql_query("SELECT * FROM student FULL OUTER JOIN course ON student.course_id = course.id", conn))
except Exception as e:
    print("Phiên bản SQLite hiện tại không hỗ trợ FULL OUTER JOIN:", e)

--- 1. TÍCH DECARTES (CROSS JOIN) ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,None,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,None,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,None,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20,7.2,12,Giai tich



--- 2. INNER JOIN ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke



--- 3. LEFT JOIN ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,3,Pham Van Nam,Toan Tin,None,7.9,None,None
3,4,Le Thanh Huyen,Toan Tin,20,7.2,None,None
4,5,Vu Quoc Anh,May Tinh,24,8.0,None,None
5,6,Dang Thuy Linh,May Tinh,24,5.5,None,None
6,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20,8.8,None,None
8,9,Duong Huu Phuc,Kinh Te,None,7.2,None,None
9,10,Cao Thi Hanh,May Tinh,None,7.0,None,None



--- 4. RIGHT JOIN ---
Phiên bản SQLite hiện tại không hỗ trợ RIGHT JOIN: Execution failed on sql 'SELECT * FROM student RIGHT JOIN course ON student.course_id = course.id': RIGHT and FULL OUTER JOINs are not currently supported

--- 5. FULL OUTER JOIN ---
Phiên bản SQLite hiện tại không hỗ trợ FULL OUTER JOIN: Execution failed on sql 'SELECT * FROM student FULL OUTER JOIN course ON student.course_id = course.id': RIGHT and FULL OUTER JOINs are not currently supported


#2.

In [4]:
# Cập nhật giá trị course_id bị thiếu thành '26' (Tin học)
cursor.execute("UPDATE student SET course_id = '26' WHERE course_id IS NULL")

# Loại bỏ các bản ghi tham gia môn học không tồn tại trong bảng course
cursor.execute("DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course)")
conn.commit()

print("--- 2a. Tổng số sinh viên và điểm trung bình từng lớp ---")
query_2a = '''
SELECT class, COUNT(student_id) AS tong_sinh_vien, ROUND(AVG(score), 2) AS diem_trung_binh
FROM student
GROUP BY class
'''
display(pd.read_sql_query(query_2a, conn))

print("\n--- 2b & 2c. Tổng số SV, Điểm TB và Phân loại thi đua từng môn học ---")
query_2bc = '''
SELECT
    course.course_name,
    COUNT(student.student_id) AS tong_sinh_vien,
    ROUND(AVG(student.score), 2) AS diem_tb,
    CASE
        WHEN AVG(student.score) >= 9.0 THEN 'Xuất sắc'
        WHEN AVG(student.score) >= 6.0 AND AVG(student.score) <= 8.9 THEN 'Tốt'
        ELSE 'Kém'
    END AS phan_loai
FROM student
JOIN course ON student.course_id = course.id
GROUP BY student.course_id
'''
display(pd.read_sql_query(query_2bc, conn))

--- 2a. Tổng số sinh viên và điểm trung bình từng lớp ---


,class,tong_sinh_vien,diem_trung_binh
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90



--- 2b & 2c. Tổng số SV, Điểm TB và Phân loại thi đua từng môn học ---


,course_name,tong_sinh_vien,diem_tb,phan_loai
0,Giai tich,1,6.70,Tốt
1,Tin hoc,3,7.37,Tốt
2,Thong ke,2,9.20,Xuất sắc


#3.

In [5]:
def hien_thi_top_bottom(df, rank_col, title):
    print(f"--- {title} ---")
    print("Top 3 cao nhất:")
    display(df[df[rank_col] <= 3].sort_values(rank_col))

    print("Top 3 thấp nhất:")
    max_rank = df[rank_col].max()
    display(df[df[rank_col] >= max_rank - 2].sort_values(rank_col, ascending=False))
    print("-" * 50)

# 3a. Xếp hạng theo điểm số
df_3a = pd.read_sql_query("SELECT *, RANK() OVER (ORDER BY score DESC) AS rank_tong FROM student", conn)
hien_thi_top_bottom(df_3a, 'rank_tong', "Xếp hạng theo ĐIỂM SỐ CHUNG")

# 3b. Xếp hạng theo lớp học
df_3b = pd.read_sql_query("SELECT *, RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_lop FROM student", conn)
print("XẾP HẠNG THEO LỚP HỌC (Ví dụ minh họa lớp 'Kinh Te'):")
hien_thi_top_bottom(df_3b[df_3b['class'] == 'Kinh Te'], 'rank_lop', "Lớp Kinh Te")

# 3c. Xếp hạng theo môn học
df_3c = pd.read_sql_query("SELECT *, RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_mon FROM student", conn)
print("XẾP HẠNG THEO MÔN HỌC (Ví dụ minh họa môn '26' - Tin hoc):")
hien_thi_top_bottom(df_3c[df_3c['course_id'] == '26'], 'rank_mon', "Môn 26 (Tin hoc)")

--- Xếp hạng theo ĐIỂM SỐ CHUNG ---
Top 3 cao nhất:


,student_id,name,class,course_id,score,rank_tong
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3


Top 3 thấp nhất:


,student_id,name,class,course_id,score,rank_tong
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4


--------------------------------------------------
XẾP HẠNG THEO LỚP HỌC (Ví dụ minh họa lớp 'Kinh Te'):
--- Lớp Kinh Te ---
Top 3 cao nhất:


...

--------------------------------------------------
XẾP HẠNG THEO MÔN HỌC (Ví dụ minh họa môn '26' - Tin hoc):
--- Môn 26 (Tin hoc) ---
Top 3 cao nhất:


,student_id,name,class,course_id,score,rank_mon
1,3,Pham Van Nam,Toan Tin,26,7.9,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,10,Cao Thi Hanh,May Tinh,26,7.0,3


Top 3 thấp nhất:


,student_id,name,class,course_id,score,rank_mon
3,10,Cao Thi Hanh,May Tinh,26,7.0,3
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
1,3,Pham Van Nam,Toan Tin,26,7.9,1


--------------------------------------------------


#4.

In [6]:
print("--- 4. BỔ SUNG THỜI GIAN TỐT NGHIỆP ---")

# Thêm cột graduation_date
try:
    cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
except sqlite3.OperationalError:
    pass # Bỏ qua lỗi nếu cột đã tồn tại khi chạy lại code nhiều lần

# Cập nhật thời gian tốt nghiệp = Thời gian hiện tại + [Hạng của SV] (ngày)
update_query = '''
WITH RankedStudents AS (
    SELECT student_id, RANK() OVER(ORDER BY score DESC) as rnk
    FROM student
)
UPDATE student
SET graduation_date = datetime('now', 'localtime', '+' || (
    SELECT rnk FROM RankedStudents WHERE RankedStudents.student_id = student.student_id
) || ' days')
'''
cursor.execute(update_query)
conn.commit()

# In kết quả cuối cùng
query_final = "SELECT student_id, name, score, graduation_date FROM student ORDER BY score DESC"
display(pd.read_sql_query(query_final, conn))

# Đóng kết nối khi hoàn thành bài tập
conn.close()

--- 4. BỔ SUNG THỜI GIAN TỐT NGHIỆP ---


,student_id,name,score,graduation_date
0,2,Tran Thi Lan,9.2,2026-04-04 13:43:04
1,7,Bui Tien Dung,9.2,2026-04-04 13:43:04
2,3,Pham Van Nam,7.9,2026-04-06 13:43:04
3,9,Duong Huu Phuc,7.2,2026-04-07 13:43:04
4,10,Cao Thi Hanh,7.0,2026-04-08 13:43:04
5,1,Nguyen Minh Hoang,6.7,2026-04-09 13:43:04
